# 🗣️ Project 9 — DebateNet: Multi-Agent Debate System

**Core Concept:** Multiple agents debate a question, critic selects strongest reasoning

### Architecture
Question → Agent A + Agent B + Agent C
              │
         Cross Review
              │
           Critic
              │
        Final Answer

Install

In [1]:
!pip install -q langchain langchain-groq langchain-core loguru

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.0 MB/s eta 0:00:00


API Key

In [2]:
import os
os.environ["GROQ_API_KEY"] = "your_groq_api_key_here"

All Setup In One Block

In [3]:
import os
import json
from datetime import datetime, timezone
from loguru import logger
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
import sys

logger.remove()
logger.add(sys.stdout, format="{time:HH:mm:ss} | {level} | {message}", level="DEBUG")

# ── Debate Agents ────────────────────────────────────────────
class DebateAgent:
    def __init__(self, name: str, persona: str, llm):
        self.name = name
        self.persona = persona
        self.llm = llm
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", f"""You are {name}, a debate agent with this perspective:
{persona}

When answering questions:
- Argue from your specific perspective
- Be clear and logical
- Support your argument with reasoning
- Keep response under 150 words
- Be direct and confident"""),
            ("human", "Question: {question}\n\nProvide your argument:")
        ])
        self.chain = self.prompt | self.llm

    def argue(self, question: str) -> dict:
        logger.info(f"{self.name} arguing...")
        response = self.chain.invoke({"question": question})
        return {
            "agent": self.name,
            "persona": self.persona,
            "argument": response.content
        }

    def review(self, question: str, other_arguments: list) -> dict:
        other_args_text = "\n\n".join([
            f"{arg['agent']}: {arg['argument']}"
            for arg in other_arguments
        ])

        review_prompt = ChatPromptTemplate.from_messages([
            ("system", f"""You are {self.name} reviewing other agents' arguments.
Your perspective: {self.persona}
Be critical but fair. Point out weaknesses in others' arguments.
Keep response under 100 words."""),
            ("human", f"""Question: {{question}}

Other agents' arguments:
{other_args_text}

Your review and counter-arguments:""")
        ])
        chain = review_prompt | self.llm
        response = chain.invoke({"question": question})
        return {
            "agent": self.name,
            "review": response.content
        }


# ── Critic Agent ─────────────────────────────────────────────
class CriticAgent:
    def __init__(self, llm):
        self.llm = llm
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", """You are an impartial critic evaluating a debate.
Your job is to:
1. Evaluate all arguments objectively
2. Identify the strongest reasoning
3. Synthesize the best points
4. Provide a final balanced answer

Be fair, analytical, and concise. Final answer under 200 words."""),
            ("human", """Question: {question}

Arguments from debate agents:
{arguments}

Reviews and counter-arguments:
{reviews}

Provide your critical evaluation and final answer:""")
        ])
        self.chain = self.prompt | self.llm

    def evaluate(self, question: str, arguments: list, reviews: list) -> dict:
        logger.info("Critic evaluating debate...")

        args_text = "\n\n".join([
            f"[{arg['agent']}]: {arg['argument']}"
            for arg in arguments
        ])

        reviews_text = "\n\n".join([
            f"[{rev['agent']} review]: {rev['review']}"
            for rev in reviews
        ])

        response = self.chain.invoke({
            "question": question,
            "arguments": args_text,
            "reviews": reviews_text
        })

        return {
            "critic_evaluation": response.content,
            "agents_involved": [arg["agent"] for arg in arguments],
            "total_arguments": len(arguments),
            "total_reviews": len(reviews)
        }


# ── DebateNet System ─────────────────────────────────────────
class DebateNetSystem:
    def __init__(self):
        self.llm = ChatGroq(
            model="llama-3.3-70b-versatile",
            temperature=0.7,
            api_key=os.environ["GROQ_API_KEY"]
        )

        self.agents = [
            DebateAgent(
                "Optimist",
                "You focus on opportunities, benefits, and positive outcomes. You believe in progress and innovation.",
                self.llm
            ),
            DebateAgent(
                "Skeptic",
                "You question assumptions, identify risks, and challenge claims. You demand evidence and highlight downsides.",
                self.llm
            ),
            DebateAgent(
                "Pragmatist",
                "You focus on practical implementation, real-world constraints, and actionable solutions. You balance idealism with reality.",
                self.llm
            )
        ]

        self.critic = CriticAgent(self.llm)
        logger.info(f"DebateNet initialized with {len(self.agents)} agents + critic")

    def debate(self, question: str) -> dict:
        logger.info(f"Starting debate: {question[:60]}")
        start_time = datetime.now(timezone.utc)

        # Round 1 — Each agent argues
        logger.info("Round 1: Initial arguments")
        arguments = []
        for agent in self.agents:
            argument = agent.argue(question)
            arguments.append(argument)
            logger.info(f"{agent.name} argument complete")

        # Round 2 — Each agent reviews others
        logger.info("Round 2: Cross reviews")
        reviews = []
        for i, agent in enumerate(self.agents):
            other_arguments = [arg for j, arg in enumerate(arguments) if j != i]
            review = agent.review(question, other_arguments)
            reviews.append(review)
            logger.info(f"{agent.name} review complete")

        # Round 3 — Critic evaluates
        logger.info("Round 3: Critic evaluation")
        verdict = self.critic.evaluate(question, arguments, reviews)

        end_time = datetime.now(timezone.utc)
        duration = (end_time - start_time).total_seconds()

        return {
            "question": question,
            "arguments": arguments,
            "reviews": reviews,
            "verdict": verdict,
            "duration_seconds": round(duration, 2),
            "timestamp": start_time.isoformat()
        }

    def display_result(self, result: dict):
        print("\n" + "="*60)
        print("DEBATENET RESULT")
        print("="*60)
        print(f"Question: {result['question']}")
        print(f"Duration: {result['duration_seconds']}s")

        print("\n--- ROUND 1: ARGUMENTS ---")
        for arg in result['arguments']:
            print(f"\n[{arg['agent']}]")
            print(arg['argument'])

        print("\n--- ROUND 2: REVIEWS ---")
        for rev in result['reviews']:
            print(f"\n[{rev['agent']} reviews others]")
            print(rev['review'])

        print("\n--- ROUND 3: CRITIC VERDICT ---")
        print(result['verdict']['critic_evaluation'])
        print("="*60)

system = DebateNetSystem()
print("DebateNet system ready")

07:56:21 | INFO | DebateNet initialized with 3 agents + critic
DebateNet system ready


Debate 1: AI in Healthcare

In [4]:
result = system.debate(
    "Should AI be used to make medical diagnoses without human doctor involvement?"
)
system.display_result(result)

07:56:21 | INFO | Starting debate: Should AI be used to make medical diagnoses without human do
07:56:21 | INFO | Round 1: Initial arguments
07:56:21 | INFO | Optimist arguing...
07:56:22 | INFO | Optimist argument complete
07:56:22 | INFO | Skeptic arguing...
07:56:22 | INFO | Skeptic argument complete
07:56:22 | INFO | Pragmatist arguing...
07:56:23 | INFO | Pragmatist argument complete
07:56:23 | INFO | Round 2: Cross reviews
07:56:23 | INFO | Optimist review complete
07:56:24 | INFO | Skeptic review complete
07:56:24 | INFO | Pragmatist review complete
07:56:24 | INFO | Round 3: Critic evaluation
07:56:24 | INFO | Critic evaluating debate...

DEBATENET RESULT
Question: Should AI be used to make medical diagnoses without human doctor involvement?
Duration: 4.26s

--- ROUND 1: ARGUMENTS ---

[Optimist]
AI should be used to make medical diagnoses without human doctor involvement because it offers immense benefits. AI systems can analyze vast amounts of data quickly and accurately, red

Debate 2: Remote Work

In [5]:
result = system.debate(
    "Is remote work better than office work for software engineers?"
)
system.display_result(result)

07:56:25 | INFO | Starting debate: Is remote work better than office work for software engineer
07:56:25 | INFO | Round 1: Initial arguments
07:56:25 | INFO | Optimist arguing...
07:56:26 | INFO | Optimist argument complete
07:56:26 | INFO | Skeptic arguing...
07:56:26 | INFO | Skeptic argument complete
07:56:26 | INFO | Pragmatist arguing...
07:56:27 | INFO | Pragmatist argument complete
07:56:27 | INFO | Round 2: Cross reviews
07:56:27 | INFO | Optimist review complete
07:56:28 | INFO | Skeptic review complete
07:56:28 | INFO | Pragmatist review complete
07:56:28 | INFO | Round 3: Critic evaluation
07:56:28 | INFO | Critic evaluating debate...

DEBATENET RESULT
Question: Is remote work better than office work for software engineers?
Duration: 3.7s

--- ROUND 1: ARGUMENTS ---

[Optimist]
Remote work is superior to office work for software engineers. It offers flexibility, autonomy, and increased productivity. Without office distractions, engineers can focus on complex coding tasks, le

Debate 3: AI Safety

In [6]:
result = system.debate(
    "Should AI development be slowed down to prioritize safety research?"
)
system.display_result(result)

07:56:29 | INFO | Starting debate: Should AI development be slowed down to prioritize safety re
07:56:29 | INFO | Round 1: Initial arguments
07:56:29 | INFO | Optimist arguing...
07:56:30 | INFO | Optimist argument complete
07:56:30 | INFO | Skeptic arguing...
07:56:30 | INFO | Skeptic argument complete
07:56:30 | INFO | Pragmatist arguing...
07:56:31 | INFO | Pragmatist argument complete
07:56:31 | INFO | Round 2: Cross reviews
07:56:32 | INFO | Optimist review complete
07:56:32 | INFO | Skeptic review complete
07:56:32 | INFO | Pragmatist review complete
07:56:32 | INFO | Round 3: Critic evaluation
07:56:32 | INFO | Critic evaluating debate...

DEBATENET RESULT
Question: Should AI development be slowed down to prioritize safety research?
Duration: 4.09s

--- ROUND 1: ARGUMENTS ---

[Optimist]
I strongly disagree. Slowing down AI development would hinder the potential benefits it can bring to society, such as improved healthcare, enhanced productivity, and increased efficiency. Instea

Project Summary

In [7]:
print("========== DEBATENET SUMMARY ==========\n")
print("Project      : DebateNet — Multi-Agent Debate System")
print("Author       : K Murali Krishna")
print("Model        : Groq LLaMA-3.3-70b-versatile")
print("\nDebate Agents:")
for agent in system.agents:
    print(f"  ✓ {agent.name:12} : {agent.persona[:60]}")
print("\nDebate Rounds:")
print("  Round 1 — Each agent argues their position")
print("  Round 2 — Each agent reviews and critiques others")
print("  Round 3 — Critic synthesizes final answer")
print("\nKey Capabilities:")
print("  ✓ Multiple specialized agent perspectives")
print("  ✓ Cross-agent review and critique")
print("  ✓ Impartial critic for final verdict")
print("  ✓ Full debate trace returned")
print("\nProduction Concepts Demonstrated:")
print("  ✓ Multi-agent coordination")
print("  ✓ Adversarial prompting for better answers")
print("  ✓ Consensus through debate")
print("  ✓ Critic pattern for quality control")

========== DEBATENET SUMMARY ==========

Project      : DebateNet — Multi-Agent Debate System
Author       : K Murali Krishna
Model        : Groq LLaMA-3.3-70b-versatile

Debate Agents:
  ✓ Optimist     : You focus on opportunities, benefits, and positive outcomes.
  ✓ Skeptic      : You question assumptions, identify risks, and challenge clai
  ✓ Pragmatist   : You focus on practical implementation, real-world constraint

Debate Rounds:
  Round 1 — Each agent argues their position
  Round 2 — Each agent reviews and critiques others
  Round 3 — Critic synthesizes final answer

Key Capabilities:
  ✓ Multiple specialized agent perspectives
  ✓ Cross-agent review and critique
  ✓ Impartial critic for final verdict
  ✓ Full debate trace returned

Production Concepts Demonstrated:
  ✓ Multi-agent coordination
  ✓ Adversarial prompting for better answers
  ✓ Consensus through debate
  ✓ Critic pattern for quality control
